# CS4120 Final — Part 2, Model 1: Conditioned N-gram LM (Week-2 baseline)

A **trigram** language model with **Kneser-Ney** smoothing, trained to generate D&D narration conditioned on a tagged prefix like

```
<ACT>attack <ACTOR>Thoradin <TGT>Goblin <ROLL>18 <VS>13 <HIT> <DMG>8_slashing <WPN>longsword <NARR>
```

The prefix is what `fireball_preprocess.linearize_for_ngram()` produces. At inference we feed the prefix ending in `<NARR>` and let the LM generate text until `</s>`.

**Why this exists for the rubric:** it's the Week-2 model on the team's course arc, and it gives us a concrete number to compare the neural models against (BLEU + perplexity).

**Runtime on free Colab:** ~2 minutes to train; inference is instant.

## 1. Setup

In [1]:
# nltk installed in local venv


In [2]:
import sys, os, json, math, random, re
from pathlib import Path
from collections import Counter

import nltk
nltk.download('punkt', quiet=True)

from nltk.lm import KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import ngrams, pad_sequence

sys.path.insert(0, '/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2')
# fireball_preprocess.py lives in the notebook directory
from fireball_preprocess import linearize_for_ngram

DATA_DIR = Path('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/processed')      # produced by 01_preprocess_fireball.ipynb
MODEL_DIR = Path('/Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2' + '/models/ngram'); MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 4120
random.seed(SEED)

print('Train/Dev/Test present:', [p.name for p in DATA_DIR.glob('*.jsonl')])

Train/Dev/Test present: ['train.jsonl', 'test.jsonl', 'dev.jsonl']


## 2. Build the training token stream

For each record we construct one **training sequence**:

```
<s> <s> <CONDITION_TOKENS> <NARR> narration_token_1 narration_token_2 ... </s>
```

The LM learns `P(w_t | w_{t-1}, w_{t-2})`. Because the condition tokens include structured tags (`<ACT>`, `<ACTOR>`, `<HIT>`), the narration that follows `<NARR>` is conditioned on them via the bigram/trigram context window.

We keep narration tokenization simple (whitespace + light regex for punctuation) so it matches what we'll expect at generation time.

In [3]:
# Lightweight tokenizer — keep words, split off simple punctuation.
RE_TOK = re.compile(r"\w+|[.,!?;:'\"\-]")

def tokenize_narration(s: str) -> list[str]:
    return RE_TOK.findall(s)

def record_to_token_stream(record: dict) -> list[str]:
    cond = linearize_for_ngram(record).split()     # already includes '<NARR>' as last token
    narr = tokenize_narration(record['narration'])
    return cond + narr

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train = load_jsonl(DATA_DIR/'train.jsonl')
dev   = load_jsonl(DATA_DIR/'dev.jsonl')
test  = load_jsonl(DATA_DIR/'test.jsonl')
print(f'Splits: train={len(train)}  dev={len(dev)}  test={len(test)}')

train_streams = [record_to_token_stream(r) for r in train]
print('\nExample training sequence:')
print(' '.join(train_streams[0][:40]), '...')

Splits: train=3533  dev=216  test=216

Example training sequence:
<ACT>attack <ACTOR>Val'trex <WPN>flame <NARR> " Stay back , you were warned , you should have left ! " ...


## 3. Train the trigram LM

`padded_everygram_pipeline` pads each sequence with `<s>` / `</s>` for us.

In [4]:
N = 3
train_data, vocab = padded_everygram_pipeline(N, train_streams)

lm = KneserNeyInterpolated(N)
lm.fit(train_data, vocab)

print(f'Vocabulary size: {len(lm.vocab)}')
print(f'Order:           {lm.order}')

Vocabulary size: 15394
Order:           3


## 4. Dev perplexity (narration-only)

We score **only the narration tokens**, not the condition prefix — the prefix is always given, so its probability mass is not the model's job to predict. This gives us a fair comparison to the T5 decoder, which also only generates the narration.

In [5]:
def narration_ngrams(record, n=N):
    # condition provides context; we only score the transitions *into* narration tokens.
    cond = linearize_for_ngram(record).split()
    narr = tokenize_narration(record['narration']) + ['</s>']
    full = list(pad_sequence(cond + narr, n=n, pad_left=True, left_pad_symbol='<s>', pad_right=False))
    trigrams = list(ngrams(full, n))
    # keep only trigrams whose *target* token is a narration token (i.e. index >= len(cond_padded))
    cond_len_padded = len(cond) + (n - 1)
    return trigrams[cond_len_padded - (n - 1):]   # skip trigrams whose target lies inside the condition

def corpus_perplexity(records):
    total_log_prob = 0.0
    total_tokens = 0
    for r in records:
        for ngram in narration_ngrams(r):
            ctx, w = tuple(ngram[:-1]), ngram[-1]
            p = lm.score(w, ctx)
            if p <= 0:
                p = 1e-12
            total_log_prob += math.log(p)
            total_tokens += 1
    avg_ll = total_log_prob / max(1, total_tokens)
    return math.exp(-avg_ll), total_tokens

dev_ppl,  dev_tok  = corpus_perplexity(dev)
test_ppl, test_tok = corpus_perplexity(test)
print(f'Dev  perplexity: {dev_ppl:8.2f}   ({dev_tok:,} tokens)')
print(f'Test perplexity: {test_ppl:8.2f}   ({test_tok:,} tokens)')

Dev  perplexity:  2993.08   (5,451 tokens)
Test perplexity:  2415.78   (5,950 tokens)


## 5. Generation

We sample from the LM conditioned on the prefix. Two strategies:
- **Greedy** (deterministic, easy to compare)
- **Top-k sampling** with `k=5` (more varied narration)

In [6]:
# --- Fast generation: sample from raw-count MLE with stupid-backoff ---
# KN scoring is O(vocab) per call; too slow in a generation loop. Raw counts
# give the same top-k candidates and are ~1000x cheaper.

_N = lm.order  # 3

def _candidates(ctx):
    """Return [(word, prob), ...] from the longest context that has counts."""
    for depth in range(min(len(ctx), _N - 1), 0, -1):
        sub = tuple(lm.vocab.lookup(ctx[-depth:]))
        cfd = lm.counts[depth + 1]
        if sub in cfd:
            counts = cfd[sub]
            if len(counts) > 0:
                total = sum(counts.values())
                return [(w, c / total) for w, c in counts.items()]
    # Unigram fallback — use top-200 most common tokens only.
    counts = lm.counts.unigrams
    total = sum(counts.values())
    return [(w, c / total) for w, c in counts.most_common(200)]

def generate(lm, condition_tokens, max_len=40, strategy='topk', k=5, temperature=1.0, seed=None):
    rng = random.Random(seed)
    ctx = list(condition_tokens[-(_N - 1):])
    out = []
    for _ in range(max_len):
        pairs = [(w, p) for w, p in _candidates(ctx) if p > 0 and w != '<s>']
        if not pairs:
            break
        if strategy == 'greedy':
            w = max(pairs, key=lambda x: x[1])[0]
        else:
            pairs.sort(key=lambda x: -x[1])
            pairs = pairs[:k]
            total = sum(p for _, p in pairs)
            r = rng.random() * total
            acc = 0.0
            w = pairs[-1][0]
            for word, p in pairs:
                acc += p
                if acc >= r:
                    w = word; break
        if w == '</s>':
            break
        out.append(w)
        ctx = (ctx + [w])[-(_N - 1):]
    return out

def detokenize(tokens):
    s = ' '.join(tokens)
    s = re.sub(r" ([.,!?;:'\"\-])", r"\1", s)
    return s

print('=== Generation samples from DEV ===')
for r in random.Random(0).sample(dev, k=min(4, len(dev))):
    cond = linearize_for_ngram(r).split()
    greedy = generate(lm, cond, strategy='greedy')
    topk   = generate(lm, cond, strategy='topk', seed=SEED)
    print('GOLD    :', r['narration'])
    print('GREEDY  :', detokenize(greedy))
    print('TOP-5   :', detokenize(topk))
    print('-' * 60)


=== Generation samples from DEV ===
GOLD    : Still move towards group yadda yadda, resisted slow. There we go.
GREEDY  : with the blunt end of the way of the way of the way of the way of the way of the way of the way of the way of the way of the way of the way of the way
TOP-5   : with a loud roar. The other just gets close."
------------------------------------------------------------
GOLD    : With the resources already used. It was a test of fate! As the petrification slowly gripped hold of the feline ranger of the arena!
GREEDY  : " I' m sorry, sir. I' m sorry, sir. I' m sorry, sir. I' m sorry, sir. I' m sorry, sir. I' m sorry
TOP-5   : " You are up to the ground, dead.
------------------------------------------------------------
GOLD    : The ogre laughs and disappears. "Puny elf." Arkan wakes up after being stabilized then takes a seat on the side to rest up. He had no regrets on going alone and practice. After all, how would he test his skills? Better here than out there.
GREED

## 6. Save the model + generated test-set predictions

The predictions file is what `04_evaluate.ipynb` consumes.

In [7]:
import pickle
with (MODEL_DIR/'kn_trigram.pkl').open('wb') as f:
    pickle.dump(lm, f)

pred_path = MODEL_DIR/'predictions_test.jsonl'
with pred_path.open('w') as f:
    for r in test:
        cond = linearize_for_ngram(r).split()
        tokens = generate(lm, cond, strategy='topk', seed=SEED)
        pred = detokenize(tokens)
        f.write(json.dumps({
            'turn_id':  r['turn_id'],
            'gold':     r['narration'],
            'pred':     pred,
            'action_type': r['action_type'],
        }, ensure_ascii=False) + '\n')
print('Wrote:', pred_path)
print('Records:', sum(1 for _ in pred_path.open()))

with (MODEL_DIR/'results.json').open('w') as f:
    json.dump({
        'model': 'kn_trigram',
        'n': N,
        'vocab_size': len(lm.vocab),
        'dev_perplexity':  dev_ppl,
        'test_perplexity': test_ppl,
    }, f, indent=2)
print('Wrote:', MODEL_DIR/'results.json')

Wrote: /Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/models/ngram/predictions_test.jsonl
Records: 216
Wrote: /Users/xuanhu/Library/CloudStorage/OneDrive-Personal/20260107-20260419/CS4120/dnd_project/part2/models/ngram/results.json


## 7. Error analysis — where does the N-gram struggle?

Cheap-but-useful: bucket perplexity by `action_type` and narration length.

In [8]:
from collections import defaultdict
buckets = defaultdict(lambda: [0.0, 0])  # action_type -> [sum_log_prob, token_count]
for r in dev:
    lp = 0.0; nt = 0
    for ngram in narration_ngrams(r):
        ctx, w = tuple(ngram[:-1]), ngram[-1]
        p = max(lm.score(w, ctx), 1e-12)
        lp += math.log(p); nt += 1
    buckets[r['action_type']][0] += lp
    buckets[r['action_type']][1] += nt

print(f'{"action":8} {"tokens":>8}  {"PPL":>8}')
for k, (lp, nt) in sorted(buckets.items(), key=lambda x: -x[1][1]):
    ppl = math.exp(-lp/max(1,nt))
    print(f'{k:8} {nt:8d}  {ppl:8.2f}')

action     tokens       PPL
attack       3362   2990.71
spell        1783   3165.11
save          306   2180.10


## What to take away for the report

- This is a **conditioned** language model — the prefix tags give the LM a fighting chance to produce action-appropriate text despite having no semantic understanding.
- We expect it to reproduce common phrasing ("the <weapon> swings", "<target> takes damage") well, but break on low-frequency named entities and long narration.
- Perplexity is the direct Week-2 metric; the T5 model in notebook `03` should beat it on BLEU/ROUGE but may have higher perplexity on held-out test (different scoring regimes).

Sources:
- [NLTK LM docs](https://www.nltk.org/api/nltk.lm.html)
- Kneser & Ney (1995) — *Improved backing-off for M-gram language modeling*